<a href="https://colab.research.google.com/github/harnepal-hub/ai-trading-bot/blob/main/AI_trading_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ccxt pandas pandas-ta

In [10]:
import requests
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
PAIRS = ["B-BTC_USDT", "B-ETH_USDT", "B-SOL_USDT"]
INITIAL_CAPITAL = 1200.00 # Base capital to calculate 1% risk

def fetch_history(pair, interval, limit=1000):
    url = f"https://public.coindcx.com/market_data/candles?pair={pair}&interval={interval}&limit={limit}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        if isinstance(data, dict): return pd.DataFrame()
        df = pd.DataFrame(data)
        df = df.sort_values(by='time').reset_index(drop=True)
        df['datetime'] = pd.to_datetime(df['time'], unit='ms')
        df.set_index(pd.DatetimeIndex(df["datetime"]), inplace=True)
        for col in ['open', 'high', 'low', 'close', 'volume']:
            df[col] = df[col].astype(float)
        return df
    except: return pd.DataFrame()

print("🚀 INITIALIZING DRAWDOWN ANALYZER...")
master_ledger = [] # This will track every trade chronologically
grand_won = 0
grand_lost = 0
grand_be = 0

for pair in PAIRS:
    print(f"📥 Processing {pair}...")
    df_regime = fetch_history(pair, "1h", 1000)
    time.sleep(2)
    df_exec = fetch_history(pair, "15m", 1000)
    time.sleep(2)

    if df_regime.empty or df_exec.empty: continue

    # GATE 1: 1-Hour Regime & Strength
    df_regime['EMA_20'] = df_regime['close'].ewm(span=20, adjust=False).mean()
    df_regime['EMA_50'] = df_regime['close'].ewm(span=50, adjust=False).mean()
    df_regime['ema_spread_pct'] = (df_regime['EMA_20'] - df_regime['EMA_50']).abs() / df_regime['close'] * 100
    df_regime['bull_regime'] = (df_regime['EMA_20'] > df_regime['EMA_50']) & (df_regime['ema_spread_pct'] > 0.15)
    df_regime['bear_regime'] = (df_regime['EMA_20'] < df_regime['EMA_50']) & (df_regime['ema_spread_pct'] > 0.15)
    df_regime['bull_regime_closed'] = df_regime['bull_regime'].shift(1)
    df_regime['bear_regime_closed'] = df_regime['bear_regime'].shift(1)
    df_exec['1h_bull'] = df_regime['bull_regime_closed'].reindex(df_exec.index, method='ffill')
    df_exec['1h_bear'] = df_regime['bear_regime_closed'].reindex(df_exec.index, method='ffill')

    # GATE 2: 15-Minute Pullback Indicators
    df_exec['SMA_20'] = df_exec['close'].rolling(window=20).mean()
    df_exec['STD_20'] = df_exec['close'].rolling(window=20).std()
    df_exec['BBL_20_2.0'] = df_exec['SMA_20'] - (df_exec['STD_20'] * 2)
    df_exec['BBU_20_2.0'] = df_exec['SMA_20'] + (df_exec['STD_20'] * 2)

    high_low = df_exec['high'] - df_exec['low']
    high_close = (df_exec['high'] - df_exec['close'].shift()).abs()
    low_close = (df_exec['low'] - df_exec['close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df_exec['ATRr_14'] = tr.rolling(window=14).mean()
    df_exec.dropna(inplace=True)

    # LOCAL SIMULATION
    position = None
    entry_price = 0
    sl, tp, position_size, breakeven_trigger = 0, 0, 0, 0
    stop_moved_to_be = False

    # We use a static balance to calculate fixed 1% risk for the ledger
    static_balance = INITIAL_CAPITAL

    for index, current in df_exec.iterrows():
        bb_lower, bb_upper = current['BBL_20_2.0'], current['BBU_20_2.0']

        if position is not None:
            trade_pnl = 0
            closed = False

            if position == 'LONG':
                if not stop_moved_to_be and current['high'] >= breakeven_trigger:
                    sl = entry_price
                    stop_moved_to_be = True

                if current['low'] <= sl:
                    trade_pnl = 0 if stop_moved_to_be else -((entry_price - sl) * position_size)
                    closed = True
                elif current['high'] >= tp:
                    trade_pnl = (tp - entry_price) * position_size
                    closed = True

            elif position == 'SHORT':
                if not stop_moved_to_be and current['low'] <= breakeven_trigger:
                    sl = entry_price
                    stop_moved_to_be = True

                if current['high'] >= sl:
                    trade_pnl = 0 if stop_moved_to_be else -((sl - entry_price) * position_size)
                    closed = True
                elif current['low'] <= tp:
                    trade_pnl = (entry_price - tp) * position_size
                    closed = True

            if closed:
                # Log the trade into the master ledger with its timestamp
                master_ledger.append({'time': index, 'pair': pair, 'pnl': trade_pnl})
                if trade_pnl > 0: grand_won += 1
                elif trade_pnl < 0: grand_lost += 1
                else: grand_be += 1
                position = None
            continue

        close, atr = current['close'], current['ATRr_14']
        risk_amount = static_balance * 0.01 # $12 risk per trade

        if current['1h_bull'] == True and close <= bb_lower:
            position = 'LONG'
            entry_price = close
            sl = entry_price - (atr * 2.0)
            tp = entry_price + (atr * 4.0)
            breakeven_trigger = entry_price + (atr * 2.0)
            position_size = risk_amount / (entry_price - sl)
            stop_moved_to_be = False

        elif current['1h_bear'] == True and close >= bb_upper:
            position = 'SHORT'
            entry_price = close
            sl = entry_price + (atr * 2.0)
            tp = entry_price - (atr * 4.0)
            breakeven_trigger = entry_price - (atr * 2.0)
            position_size = risk_amount / (sl - entry_price)
            stop_moved_to_be = False

# ==========================================
# MAXIMUM DRAWDOWN CALCULATION
# ==========================================
# Sort all trades from all pairs chronologically
master_ledger.sort(key=lambda x: x['time'])

simulated_balance = INITIAL_CAPITAL
peak_balance = INITIAL_CAPITAL
max_drawdown_usd = 0

for trade in master_ledger:
    simulated_balance += trade['pnl']

    # Check for new all-time high
    if simulated_balance > peak_balance:
        peak_balance = simulated_balance

    # Calculate how far we are from the peak
    current_drawdown = peak_balance - simulated_balance
    if current_drawdown > max_drawdown_usd:
        max_drawdown_usd = current_drawdown

max_drawdown_pct = (max_drawdown_usd / INITIAL_CAPITAL) * 100

print("\n==========================================")
print("🛡️ PORTFOLIO RISK & DRAWDOWN REPORT")
print("==========================================")
print(f"Total Trades Taken: {len(master_ledger)}")
print(f"Total Won:          {grand_won}")
print(f"Total Lost:         {grand_lost}")
print(f"Total Breakeven:    {grand_be}")
print(f"Net Profit:         ${(simulated_balance - INITIAL_CAPITAL):,.2f}")
print("------------------------------------------")
print(f"📉 MAX DRAWDOWN:    ${max_drawdown_usd:.2f} ({max_drawdown_pct:.2f}%)")
print("==========================================")
if max_drawdown_pct < 5.0:
    print("✅ RATING: Excellent. Very low risk to capital.")
elif max_drawdown_pct < 10.0:
    print("⚠️ RATING: Acceptable. Normal market turbulence.")
else:
    print("❌ RATING: High Risk. Consider lowering 1% risk per trade.")

🚀 INITIALIZING DRAWDOWN ANALYZER...
📥 Processing B-BTC_USDT...
📥 Processing B-ETH_USDT...
📥 Processing B-SOL_USDT...

🛡️ PORTFOLIO RISK & DRAWDOWN REPORT
Total Trades Taken: 40
Total Won:          11
Total Lost:         14
Total Breakeven:    15
Net Profit:         $96.00
------------------------------------------
📉 MAX DRAWDOWN:    $48.00 (4.00%)
✅ RATING: Excellent. Very low risk to capital.
